# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saisathwik2703/flyrank.ai_internship/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
import pandas as pd
import numpy as np
import os

# Load model predictions (generated by w05_model.ipynb)
try:
    preds = pd.read_csv('work/outputs/model_predictions.csv')
except FileNotFoundError:
    # Fall back to baseline queue if model predictions not available
    try:
        preds = pd.read_csv('work/outputs/baseline_refresh_queue.csv')
        preds['model_prob'] = 1 - (preds['rank'] / preds['rank'].max())
    except FileNotFoundError:
        print('Run w04_baseline_score.ipynb or w05_model.ipynb first.')
        import sys; sys.exit(0)

preds_sorted = preds.sort_values('model_prob', ascending=False).reset_index(drop=True)
preds_sorted['refresh_rank'] = preds_sorted.index + 1

# Generate human-readable action recommendation per page
def get_action(row):
    """Return a recommended action based on signals."""
    actions = []
    if row.get('days_since_last_update', 0) > 180:
        actions.append('REWRITE: Content is stale (>180 days since last update)')
    if row.get('avg_position', 50) > 20:
        actions.append('OPTIMIZE TITLE/META: Page is below position 20')
    if row.get('ctr', 0) < 1.0:
        actions.append('IMPROVE CTR: Low click-through rate — rewrite title and meta description')
    if row.get('engagement_rate', 100) < 40:
        actions.append('IMPROVE CONTENT DEPTH: Low engagement rate — expand and improve content quality')
    if not actions:
        actions.append('REVIEW: High decline probability — investigate recent SERP changes')
    return ' | '.join(actions)

if 'days_since_last_update' in preds_sorted.columns:
    preds_sorted['recommended_action'] = preds_sorted.apply(get_action, axis=1)
else:
    preds_sorted['recommended_action'] = 'REVIEW: Flagged as high-risk by model'

# Display top 20 with action recommendations
display_cols = ['refresh_rank', 'content_id', 'client_id', 'model_prob', 'recommended_action']
if 'is_declining_label' in preds_sorted.columns:
    display_cols.append('is_declining_label')

print('Top 20 Content Refresh Recommendations:')
print('=' * 80)
for _, row in preds_sorted.head(20).iterrows():
    print(f"#{int(row['refresh_rank'])} | {row['content_id']} | Score: {row['model_prob']:.3f}")
    if 'recommended_action' in row:
        print(f"   Action: {row['recommended_action']}")
    print()

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use

**Who:** Content managers and SEO strategists at FlyRank client companies, reviewed by a FlyRank account manager.

**For what:** Weekly/monthly sprint planning. The team opens the ranked queue, reviews the top 20–50 pages, and assigns writer slots based on the model's urgency scores and reason codes. The model does not make automatic changes — it is a decision-support tool.

**Workflow:**
1. Export the ranked queue CSV from this notebook (or from the FlyRank platform).
2. Content manager reviews each top-ranked page manually: confirm the decline signal, assess whether it is content quality vs SERP change vs seasonality.
3. Assign the page to a writer with the reason codes as the brief.
4. Writer refreshes the content; page gets `days_since_last_update` reset to 0.
5. Track outcomes over the next 60 days: did impressions recover?

---

### Where This Stops Being Valid

| Limitation | Impact | Mitigation |
|---|---|---|
| Proxy label (past trend ≠ future recovery) | High-confidence refresh may not recover traffic if decline was caused by external SERP shift | Manual review step in workflow |
| 32 client training set | May not generalize to new client industries or content niches | Re-train periodically as FlyRank adds clients |
| Seasonal content | Holiday/event content fires the label during off-season but recovers naturally | Suppress from queue if URL contains seasonal keywords |
| AI-traffic confound | Pages gaining AI traffic while losing GSC impressions may still score as declining | Add AI_traffic_growing flag to suppress if AI session growth > 50% QoQ |
| No intervention feedback loop | Model doesn't know which previously refreshed pages recovered | Track refresh outcomes; use as future positive label for training |
| Stale model | Google algorithm updates change what predicts decline | Re-train quarterly with fresh data |

## 3. Summary: what I built and what I learned

*One paragraph each.*

### What I Built

Over these 7 weeks I built a complete content refresh prioritization system on top of FlyRank's anonymized search data. Starting from a 30,000-row × 44-column dataset, I defined a binary classification task (predicting declining pages), engineered 30+ features from GSC, GA4, keyword context, and content metadata, audited for leakage and missingness, designed a client-grouped holdout split, and trained a LightGBM model. The model's Precision@50 outperforms the hand-written rule baseline. The final output is a ranked refresh queue with human-readable reason codes that a content team can act on without needing to understand the model internals.

### What I Learned

The most important lesson was about proxy labels and methodological honesty: the `is_declining_label` captures recent retrospective trend, not future outcome. This distinction matters enormously in the real world — a model that perfectly predicts the proxy label might still fail to identify the pages that will benefit most from being refreshed. The second key lesson was the client-holdout split: random row splits would have given optimistically inflated Precision@50 by letting the model memorize per-client patterns. Finally, I learned that AI traffic (`ai_traffic_pct`) does not protect pages from declining in traditional Google Search — these are genuinely separate channels, and conflating them would lead content teams to ignore declining pages that happen to be doing well on AI referrals.